# Helene Exploratory Maps (N=271 counties)

**Objective**: Visualize the spatial distribution of income and disruption
across Helene's affected region to guide the analysis strategy.

**Maps**:
1. Income level (choropleth)
2. Largest drop (within)
3. Distance to hurricane track
4. Bivariate: income + drop
5. Hurricane track overlay

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

from shapely.geometry import LineString

warnings.filterwarnings("ignore")

OUTPUT_DIR = "../results/helene_exploratory/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)
print("Setup complete.")

## 1. Load Data

In [ ]:
# ── Load Helene county list ──
with open("../results/helene/counties_geoid_cut_50.txt", "r") as f:
    helene_geoids = [int(line.strip()) for line in f if line.strip()]
print(f"Helene affected counties: {len(helene_geoids)}")

# ── Load largest drop (within) ──
drop_df = pd.read_csv("../results/helene/largest_drop_within.csv")
drop_df["GEOID"] = drop_df["GEOID"].astype(int)
print(f"Drop data: {len(drop_df)} counties")

# ── Load ACS ──
acs_df = pd.read_csv("acs_socioeconomic_v2.csv")
acs_df["GEOID"] = acs_df["GEOID"].astype(int)

# ── Load county geometries ──
county_shp = "./../../hurricane_oct/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp"
counties_gdf = gpd.read_file(county_shp)
counties_gdf["GEOID"] = counties_gdf["GEOID"].astype(int)

# Filter to Helene counties
gdf = counties_gdf[counties_gdf["GEOID"].isin(helene_geoids)].copy()
print(f"County geometries matched: {len(gdf)}")

# ── Load storm track ──
track_gdf = gpd.read_file("./../../hurricane_oct/data/storm_track/helene_storm_track.shp")
print(f"Track CRS: {track_gdf.crs}, records: {len(track_gdf)}")

In [ ]:
# ── Merge all data ──
gdf = gdf.merge(
    acs_df[["GEOID", "total_population", "median_household_income",
            "pct_no_vehicle", "insurance_coverage_pct"]],
    on="GEOID", how="left"
)
gdf = gdf.merge(drop_df[["GEOID", "largest_drop"]], on="GEOID", how="left")

# ── Project to Albers Equal Area ──
gdf = gdf.to_crs(epsg=5070)
track_proj = track_gdf.to_crs(epsg=5070)

# Build track line
if track_proj.geometry.geom_type.iloc[0] == "Point":
    track_line = LineString(track_proj.geometry.tolist())
else:
    track_line = track_proj.unary_union

# ── Compute distance to track ──
gdf["centroid"] = gdf.geometry.centroid
gdf["dist_to_track_km"] = gdf["centroid"].apply(lambda pt: track_line.distance(pt)) / 1000.0
gdf["dist_to_track_mi"] = gdf["dist_to_track_km"] / 1.60934

# ── Income quartiles ──
gdf["income_quartile"] = pd.qcut(
    gdf["median_household_income"].rank(method="first"),
    q=4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
)

print(f"\nFinal GeoDataFrame: {len(gdf)} counties")
print(f"Income range: ${gdf['median_household_income'].min():,.0f} – ${gdf['median_household_income'].max():,.0f}")
print(f"Distance range: {gdf['dist_to_track_mi'].min():.1f} – {gdf['dist_to_track_mi'].max():.1f} mi")
print(f"Drop range: {gdf['largest_drop'].min():.1f}% – {gdf['largest_drop'].max():.1f}%")
print(f"\nIncome quartile counts:")
print(gdf["income_quartile"].value_counts().sort_index())
print(f"\nMissing values:")
print(gdf[["median_household_income", "largest_drop", "dist_to_track_mi"]].isna().sum())

In [ ]:
# ── Track GeoDataFrame for plotting ──
track_line_gdf = gpd.GeoDataFrame(geometry=[track_line], crs="EPSG:5070")

# Save distance data
gdf[["GEOID", "NAME", "dist_to_track_km", "dist_to_track_mi",
     "median_household_income", "largest_drop", "income_quartile"]].to_csv(
    os.path.join(OUTPUT_DIR, "helene_county_data.csv"), index=False
)
print("Saved county data CSV.")

## 2. Map: Median Household Income

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

gdf.plot(
    column="median_household_income",
    cmap="RdYlGn",
    legend=True,
    legend_kwds={"label": "Median Household Income ($)", "shrink": 0.6},
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "No data"},
)

# Overlay track
track_line_gdf.plot(ax=ax, color="red", linewidth=3, label="Helene Track")

ax.set_title("Helene Affected Region: Median Household Income\nwith Hurricane Track",
             fontsize=14, fontweight="bold")
ax.set_axis_off()

# Legend for track
legend_elements = [Line2D([0], [0], color="red", linewidth=3, label="Helene Track")]
ax.legend(handles=legend_elements, loc="lower left", fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "map_income.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Map: Income Quartile (categorical)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

colors_q = {"Q1 (lowest)": "#d73027", "Q2": "#fc8d59",
            "Q3": "#91bfdb", "Q4 (highest)": "#4575b4"}

for q, color in colors_q.items():
    subset = gdf[gdf["income_quartile"] == q]
    subset.plot(ax=ax, color=color, edgecolor="black", linewidth=0.3, label=q)

# Missing
missing = gdf[gdf["income_quartile"].isna()]
if len(missing) > 0:
    missing.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.3, label="No data")

# Track
track_line_gdf.plot(ax=ax, color="black", linewidth=3, linestyle="--")

ax.set_title("Helene Affected Region: Income Quartiles\n(Red = lowest, Blue = highest)",
             fontsize=14, fontweight="bold")
ax.set_axis_off()
ax.legend(loc="lower left", fontsize=10, title="Income Quartile")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "map_income_quartile.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Map: Largest Drop (within)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

gdf.plot(
    column="largest_drop",
    cmap="RdBu",
    legend=True,
    legend_kwds={"label": "Largest Drop (%)", "shrink": 0.6},
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "No data"},
)

# Track
track_line_gdf.plot(ax=ax, color="black", linewidth=3, linestyle="--")

ax.set_title("Helene Affected Region: Largest Within-Region Mobility Drop (%)\n(More negative = larger disruption)",
             fontsize=14, fontweight="bold")
ax.set_axis_off()

legend_elements = [Line2D([0], [0], color="black", linewidth=3, linestyle="--", label="Helene Track")]
ax.legend(handles=legend_elements, loc="lower left", fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "map_largest_drop.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Map: Distance to Track

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

gdf.plot(
    column="dist_to_track_mi",
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "Distance to Track (mi)", "shrink": 0.6},
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
)

track_line_gdf.plot(ax=ax, color="blue", linewidth=3)

ax.set_title("Helene Affected Region: Distance to Hurricane Track",
             fontsize=14, fontweight="bold")
ax.set_axis_off()

legend_elements = [Line2D([0], [0], color="blue", linewidth=3, label="Helene Track")]
ax.legend(handles=legend_elements, loc="lower left", fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "map_distance_to_track.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Side-by-Side: Income vs Drop (2-panel)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

# Panel 1: Income
gdf.plot(column="median_household_income", cmap="RdYlGn", legend=True,
         legend_kwds={"label": "Income ($)", "shrink": 0.5},
         edgecolor="black", linewidth=0.2, ax=ax1,
         missing_kwds={"color": "lightgrey"})
track_line_gdf.plot(ax=ax1, color="black", linewidth=2.5, linestyle="--")
ax1.set_title("Median Household Income", fontsize=13, fontweight="bold")
ax1.set_axis_off()

# Panel 2: Drop
gdf.plot(column="largest_drop", cmap="RdBu", legend=True,
         legend_kwds={"label": "Drop (%)", "shrink": 0.5},
         edgecolor="black", linewidth=0.2, ax=ax2,
         missing_kwds={"color": "lightgrey"})
track_line_gdf.plot(ax=ax2, color="black", linewidth=2.5, linestyle="--")
ax2.set_title("Largest Within-Region Drop (%)", fontsize=13, fontweight="bold")
ax2.set_axis_off()

fig.suptitle("Helene: Income vs Mobility Drop (N=271 counties)",
             fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "panel_income_vs_drop.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Quick Descriptive Statistics

In [ ]:
print("="*60)
print("HELENE DESCRIPTIVE STATISTICS")
print("="*60)

print("\nIncome by quartile:")
print(gdf.groupby("income_quartile").agg(
    n=("GEOID", "count"),
    income_min=("median_household_income", "min"),
    income_max=("median_household_income", "max"),
    income_median=("median_household_income", "median"),
    drop_median=("largest_drop", "median"),
    drop_mean=("largest_drop", "mean"),
    dist_median=("dist_to_track_mi", "median"),
).round(1))

# Spearman correlations
from scipy.stats import spearmanr
valid = gdf[["median_household_income", "largest_drop", "dist_to_track_mi"]].dropna()
print(f"\nSpearman correlations (N={len(valid)}):")
for v1, v2 in [("median_household_income", "largest_drop"),
               ("median_household_income", "dist_to_track_mi"),
               ("dist_to_track_mi", "largest_drop")]:
    rho, p = spearmanr(valid[v1], valid[v2])
    sig = "**" if p < 0.05 else "*" if p < 0.1 else ""
    print(f"  {v1} vs {v2}: ρ = {rho:.3f}, p = {p:.4f} {sig}")

# Income–distance confound check for Helene
print(f"\nMedian distance to track by income quartile:")
print(gdf.groupby("income_quartile")["dist_to_track_mi"].median().round(1))

## 8. Scatter: Drop vs Income (colored by distance)

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess as lowess_func

valid = gdf[["median_household_income", "largest_drop", "dist_to_track_mi"]].dropna()

fig, ax = plt.subplots(figsize=(12, 7))

norm = mcolors.Normalize(vmin=valid["dist_to_track_mi"].min(),
                         vmax=valid["dist_to_track_mi"].max())
sc = ax.scatter(valid["median_household_income"], valid["largest_drop"],
                c=valid["dist_to_track_mi"], cmap="RdYlGn_r", norm=norm,
                s=40, edgecolors="black", linewidth=0.3, alpha=0.8)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Distance to Track (mi)", fontsize=11)

# LOWESS
lw = lowess_func(valid["largest_drop"].values, valid["median_household_income"].values, frac=0.3)
ax.plot(lw[:, 0], lw[:, 1], color="blue", linewidth=2.5, label="LOWESS (frac=0.3)")

# OLS
z = np.polyfit(valid["median_household_income"], valid["largest_drop"], 1)
x_range = np.linspace(valid["median_household_income"].min(), valid["median_household_income"].max(), 100)
ax.plot(x_range, np.polyval(z, x_range), color="gray", ls="--", lw=1.5, label="OLS fit")

from scipy.stats import spearmanr, pearsonr
r_p, p_p = pearsonr(valid["median_household_income"], valid["largest_drop"])
r_s, p_s = spearmanr(valid["median_household_income"], valid["largest_drop"])
ax.text(0.02, 0.98, f"Pearson r={r_p:.3f} (p={p_p:.3f})\nSpearman ρ={r_s:.3f} (p={p_s:.3f})",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round,pad=0.3", fc="wheat", alpha=0.5))

ax.set_xlabel("Median Household Income ($)", fontsize=12)
ax.set_ylabel("Largest Drop — Within (%)", fontsize=12)
ax.set_title("Helene: Mobility Drop vs Income (N=271, colored by track distance)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figures", "scatter_drop_vs_income.png"), dpi=150, bbox_inches="tight")
plt.show()